# Explainable Credit Risk Prediction System
### A complete ML pipeline: EDA → Preprocessing → Model Comparison → SHAP Explainability

---

**Dataset:** German Credit (UCI / Kaggle) — 1,000 applicants, 9 features  
**Goal:** Predict whether a loan applicant will default (`bad`) or repay (`good`)  
**Key Technology:** XGBoost + SHAP (SHapley Additive exPlanations)  

---

## Notebook Structure

| Section | What You'll Learn |
|---------|-------------------|
| 1. Setup | Load libraries and data |
| 2. EDA | Understand the dataset, distributions, class balance |
| 3. Preprocessing | Encoding, scaling, data splitting |
| 4. Model Training | Train LR, Random Forest, XGBoost and compare fairly |
| 5. Evaluation | ROC-AUC, Gini, F1, confusion matrix |
| 6. SHAP Explanations | Global importance + per-applicant explanations |
| 7. API Demo | Test the live prediction API |

---
## Section 1 — Setup & Imports

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import shap

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    classification_report, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay
)
import xgboost as xgb

# Add src/ to path so we can import our own modules
sys.path.insert(0, os.path.join('..', 'src'))
from preprocessing import (
    load_data, split_data, build_preprocessor,
    get_feature_names, prepare_data,
    FILL_NONE_COLS, ORDINAL_COLS, ONEHOT_COLS, NUMERIC_COLS, PASSTHROUGH_COLS
)

# Matplotlib style
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   'white',
    'font.family':      'DejaVu Sans',
    'axes.spines.top':  False,
    'axes.spines.right':False,
})

SEED       = 42
DATA_PATH  = os.path.join('..', 'data', 'raw', 'german_credit_with_risk.csv')
MODEL_DIR  = os.path.join('..', 'models')
REPORT_DIR = os.path.join('..', 'reports')

print('All libraries loaded successfully ✓')
print(f'pandas  : {pd.__version__}')
print(f'numpy   : {np.__version__}')
print(f'xgboost : {xgb.__version__}')
print(f'shap    : {shap.__version__}')

---
## Section 2 — Exploratory Data Analysis (EDA)

Before building any model, we must understand the data. EDA answers:
- What does the dataset look like?
- Are there missing values?
- Is the target class balanced?
- Which features might matter most?

In [ ]:
# ── 2.1 Load raw data ──────────────────────────────────────────────────────
df_raw = pd.read_csv(DATA_PATH, index_col=0)

print(f'Shape: {df_raw.shape}  ({df_raw.shape[0]} rows × {df_raw.shape[1]} columns)')
print()
print('Column types:')
print(df_raw.dtypes)
print()
df_raw.head(10)

In [ ]:
# ── 2.2 Missing values ─────────────────────────────────────────────────────
# Key insight: 'Saving accounts' and 'Checking account' have missing values
# These are INFORMATIVE missings — missing likely means 'no account'

missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(1)

missing_df = pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})
missing_df = missing_df[missing_df['Missing Count'] > 0]

print('Columns with missing values:')
print(missing_df.to_string())
print()
print("Strategy: Fill NaN with 'none' (meaning: no account)")
print("Reason: 'none' is genuinely a credit risk signal — not random noise")

In [ ]:
# ── 2.3 Target distribution ────────────────────────────────────────────────
# CRITICAL: Is this a balanced or imbalanced classification problem?

target_counts = df_raw['Risk'].value_counts()
target_pct    = df_raw['Risk'].value_counts(normalize=True) * 100

print('Class distribution:')
print(f"  good (will repay)  : {target_counts['good']:4d}  ({target_pct['good']:.1f}%)")
print(f"  bad  (will default): {target_counts['bad']:4d}  ({target_pct['bad']:.1f}%)")
print()
print('Class imbalance ratio:', round(target_counts['good'] / target_counts['bad'], 2), ':1')
print()
print('Implication: We CANNOT use accuracy as a metric.')
print('  A model that always predicts "good" gets 70% accuracy — but catches 0 defaulters!')
print('  We use ROC-AUC and Recall(bad) instead.')

fig, ax = plt.subplots(figsize=(5, 4))
colors = ['#4caf50', '#f44336']
bars = ax.bar(target_counts.index, target_counts.values, color=colors, width=0.5, edgecolor='white')
for bar, pct in zip(bars, target_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8,
            f'{pct:.1f}%', ha='center', fontweight='bold', fontsize=12)
ax.set_title('Target Class Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Applicants')
ax.set_ylim(0, 820)
plt.tight_layout()
plt.show()

In [ ]:
# ── 2.4 Numeric feature distributions by Risk class ────────────────────────
# Shows whether numeric features separate the two classes

numeric_features = ['Age', 'Credit amount', 'Duration']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, numeric_features):
    good_vals = df_raw[df_raw['Risk'] == 'good'][col]
    bad_vals  = df_raw[df_raw['Risk'] == 'bad'][col]
    
    ax.hist(good_vals, bins=25, alpha=0.6, color='#4caf50', label='Good')
    ax.hist(bad_vals,  bins=25, alpha=0.6, color='#f44336', label='Bad')
    ax.set_title(col, fontweight='bold')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.legend()

    # Print stats
    print(f"--- {col} ---")
    print(f"  Good: mean={good_vals.mean():.1f}, median={good_vals.median():.1f}")
    print(f"  Bad:  mean={bad_vals.mean():.1f}, median={bad_vals.median():.1f}")
    print()

plt.suptitle('Numeric Features: Good vs Bad Applicants', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Key insight: Bad applicants tend to have higher Credit Amount and longer Duration')

In [ ]:
# ── 2.5 Categorical feature analysis ───────────────────────────────────────
# Which categories are associated with higher default rates?

categorical_features = ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(categorical_features):
    ax = axes[i]
    
    # Fill NaN so we can analyse the 'none' category too
    temp = df_raw[['Risk', col]].copy()
    temp[col] = temp[col].fillna('none')
    
    # Calculate bad rate per category
    grouped = temp.groupby(col)['Risk'].apply(
        lambda x: (x == 'bad').sum() / len(x) * 100
    ).sort_values(ascending=False)
    
    bars = ax.bar(grouped.index, grouped.values,
                  color=[plt.cm.RdYlGn_r(v/100) for v in grouped.values],
                  edgecolor='white')
    ax.set_title(f'{col}\n(Default Rate %)', fontweight='bold', fontsize=10)
    ax.set_ylabel('Default Rate (%)')
    ax.tick_params(axis='x', rotation=30, labelsize=8)
    ax.set_ylim(0, 75)
    
    for bar, val in zip(bars, grouped.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                f'{val:.0f}%', ha='center', fontsize=7.5, fontweight='bold')

axes[-1].set_visible(False)
plt.suptitle('Default Rate by Category — Which groups are riskier?',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print('Key insights:')
print('  - No Checking account → highest default rate (~55%)')
print('  - No Saving account   → high default rate')
print('  - Education loans     → high default rate')
print('  - Rich account holders → very low default rate')

In [ ]:
# ── 2.6 Correlation heatmap (numeric features) ─────────────────────────────

df_corr = df_raw[['Age', 'Credit amount', 'Duration', 'Job']].copy()
df_corr['Risk_binary'] = (df_raw['Risk'] == 'bad').astype(int)

fig, ax = plt.subplots(figsize=(6, 5))
corr_matrix = df_corr.corr()
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5,
            cbar_kws={'shrink': 0.8}, ax=ax)
ax.set_title('Correlation Matrix (Numeric Features + Target)', fontweight='bold')
plt.tight_layout()
plt.show()

print('Duration and Credit amount are positively correlated (0.62)')
print('Longer loans tend to be for larger amounts — makes sense')

In [ ]:
# ── 2.7 Summary statistics ─────────────────────────────────────────────────
print("=" * 50)
print("DATASET SUMMARY")
print("=" * 50)
print(df_raw.describe().to_string())

---
## Section 3 — Preprocessing

The **golden rule of preprocessing:** fit encoders/scalers **only on training data**, then apply to validation and test data. This prevents **data leakage** — where future information contaminates the model.

Our pipeline:
1. **Fill missing values** with `'none'` (informative missingness)
2. **Encode target**: `good → 0`, `bad → 1`
3. **Split**: 70% train / 15% validation / 15% test (stratified)
4. **Transform features**:
   - `OrdinalEncoder` → `Saving accounts`, `Checking account` (ordered scale)
   - `OneHotEncoder` → `Sex`, `Housing`, `Purpose` (no natural order)
   - `StandardScaler` → `Age`, `Credit amount`, `Duration`
   - **Passthrough** → `Job` (already numeric 0–3)

In [ ]:
# ── 3.1 Full preprocessing pipeline ───────────────────────────────────────

(X_train, X_val, X_test,
 y_train, y_val, y_test,
 preprocessor, feature_names) = prepare_data(DATA_PATH)

print(f'\nProcessed data shapes:')
print(f'  X_train : {X_train.shape}')
print(f'  X_val   : {X_val.shape}')
print(f'  X_test  : {X_test.shape}')

In [ ]:
# ── 3.2 View feature names after encoding ─────────────────────────────────
# OneHotEncoder expands 3 categorical columns into multiple binary columns

print('Feature names after all transformations:')
print(f'Total: {len(feature_names)} features')
print()

for group, cols in [
    ('Ordinal encoded (account levels)', ORDINAL_COLS),
    ('One-hot encoded (binary flags)',   [f for f in feature_names if any(c in f for c in ONEHOT_COLS)]),
    ('Scaled numeric',                   NUMERIC_COLS),
    ('Passthrough (unchanged)',          PASSTHROUGH_COLS),
]:
    print(f'  [{group}]')
    for col in cols:
        print(f'    • {col}')
    print()

print()
print('Sample of processed training data (first 3 rows):')
pd.DataFrame(X_train[:3], columns=feature_names).round(3)

In [ ]:
# ── 3.3 Class balance check after split ───────────────────────────────────
# Stratified split ensures all splits have similar class ratios

print('Class balance in each split (stratified split preserves the original ratio):')
print(f'  Train bad%: {y_train.mean():.1%}  ({y_train.sum()} bad, {(1-y_train).sum()} good)')
print(f'  Val   bad%: {y_val.mean():.1%}  ({y_val.sum()} bad, {(1-y_val).sum()} good)')
print(f'  Test  bad%: {y_test.mean():.1%}  ({y_test.sum()} bad, {(1-y_test).sum()} good)')
print()
print('Good: each split has approximately the same 30% bad rate.')
print('This is guaranteed by stratify=y in train_test_split().')

---
## Section 4 — Model Training

We train three models to demonstrate the **progression from simple to complex**:

| Model | Why we include it |
|-------|-------------------|
| Logistic Regression | Baseline — linear, fully interpretable |
| Random Forest | Intermediate — captures non-linear patterns |
| XGBoost | Primary — gradient boosting, best performance |

### Handling class imbalance
All models receive special treatment for the 70/30 imbalance:
- LR & RF: `class_weight='balanced'` — sklearn automatically adjusts sample weights
- XGBoost: `scale_pos_weight = 700/300 ≈ 2.33` — tells XGBoost to weight bad applicants 2.33× more

In [ ]:
# ── 4.1 Define models ─────────────────────────────────────────────────────

models = {
    'Logistic Regression': LogisticRegression(
        class_weight='balanced',
        max_iter=1000,
        random_state=SEED
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        class_weight='balanced',
        random_state=SEED,
        n_jobs=-1
    ),
    'XGBoost': xgb.XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        scale_pos_weight=700/300,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=1.0,
        eval_metric='auc',
        use_label_encoder=False,
        random_state=SEED,
        verbosity=0
    ),
}

print('Models defined:')
for name in models:
    print(f'  • {name}')

In [ ]:
# ── 4.2 Train all models ──────────────────────────────────────────────────

trained_models = {}
results = []

for name, model in models.items():
    print(f'Training {name}...')
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f'  Done ✓')

print('\nAll models trained successfully!')

---
## Section 5 — Model Evaluation

### Why ROC-AUC?
- **Accuracy** is useless for imbalanced datasets. A model that always predicts "good" gets 70% accuracy but is useless.
- **ROC-AUC** measures how well the model **ranks** applicants by risk. 1.0 = perfect, 0.5 = random guessing.
- **Gini** = 2 × AUC − 1. Industry-standard credit risk metric. 0 = random, 1 = perfect.
- **Recall(bad)** = % of actual defaulters we correctly catch. Critical for banks.
- **Precision(bad)** = of those we flag as bad, how many truly are. Affects false rejections.

In [ ]:
# ── 5.1 Compute all metrics ────────────────────────────────────────────────

results = []

for name, model in trained_models.items():
    val_proba  = model.predict_proba(X_val)[:,1]
    test_proba = model.predict_proba(X_test)[:,1]
    test_pred  = (test_proba >= 0.5).astype(int)

    val_auc   = roc_auc_score(y_val, val_proba)
    test_auc  = roc_auc_score(y_test, test_proba)
    val_prauc = average_precision_score(y_val, val_proba)
    gini      = 2 * test_auc - 1

    report = classification_report(y_test, test_pred,
                                   target_names=['good','bad'], output_dict=True)

    results.append({
        'Model':          name,
        'Val AUC':        round(val_auc, 4),
        'Test AUC':       round(test_auc, 4),
        'Gini':           round(gini, 4),
        'Val PR-AUC':     round(val_prauc, 4),
        'Recall(bad)':    round(report['bad']['recall'], 4),
        'Precision(bad)': round(report['bad']['precision'], 4),
        'F1(bad)':        round(report['bad']['f1-score'], 4),
    })

df_results = pd.DataFrame(results)
df_results.set_index('Model', inplace=True)

print('=== MODEL COMPARISON TABLE ===')
print(df_results.to_string())
print()
print('Winner: XGBoost — highest AUC, Gini, and F1(bad)')

In [ ]:
# ── 5.2 Style the comparison table ────────────────────────────────────────

df_display = df_results.copy()
df_display.style.highlight_max(
    subset=['Test AUC', 'Gini', 'Recall(bad)', 'F1(bad)'],
    color='#c8e6c9'
).format('{:.4f}').set_caption('Model Comparison — Green = Best in column')

In [ ]:
# ── 5.3 ROC Curves — all three models ────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

colors = ['#1976d2', '#388e3c', '#d32f2f']

for ax, plot_fn, title in [
    (axes[0], RocCurveDisplay.from_predictions, 'ROC Curves'),
    (axes[1], PrecisionRecallDisplay.from_predictions, 'Precision-Recall Curves'),
]:
    for (name, model), color in zip(trained_models.items(), colors):
        proba = model.predict_proba(X_test)[:, 1]
        plot_fn(y_test, proba, name=name, ax=ax, color=color)
    ax.set_title(title, fontweight='bold')

# Add random baseline diagonal for ROC
axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.4, label='Random (AUC=0.5)')
axes[0].legend(loc='lower right')
axes[1].legend(loc='upper right')

plt.suptitle('Model Performance Comparison on Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'roc_pr_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reports/roc_pr_curves.png')

In [ ]:
# ── 5.4 Confusion matrices ────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (name, model) in zip(axes, trained_models.items()):
    y_pred = (model.predict_proba(X_test)[:, 1] >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Pred Good', 'Pred Bad'],
                yticklabels=['True Good', 'True Bad'],
                ax=ax, cbar=False, linewidths=0.5)
    ax.set_title(name, fontweight='bold')

    tn, fp, fn, tp = cm.ravel()
    recall = tp / (tp + fn)
    ax.set_xlabel(f'Recall(bad)={recall:.0%}  |  TP={tp}  FN={fn}', fontsize=9)

plt.suptitle('Confusion Matrices (Test Set, threshold=0.5)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Key metric: Recall(bad) = % of actual defaulters we catch')
print('FN (bottom-left) = bad applicants we wrongly approved — most dangerous for banks')

In [ ]:
# ── 5.5 Prediction probability distributions ──────────────────────────────
# Shows how well the model separates the two classes

model = trained_models['XGBoost']
test_proba = model.predict_proba(X_test)[:, 1]

fig, ax = plt.subplots(figsize=(8, 4))

good_mask = y_test == 0
bad_mask  = y_test == 1

ax.hist(test_proba[good_mask], bins=25, alpha=0.6, color='#4caf50', label='True Good')
ax.hist(test_proba[bad_mask],  bins=25, alpha=0.6, color='#f44336', label='True Bad')
ax.axvline(0.5, color='black', linestyle='--', linewidth=1.5, label='Decision boundary (0.5)')

ax.set_xlabel('Predicted Probability of Default', fontsize=11)
ax.set_ylabel('Count', fontsize=11)
ax.set_title('XGBoost — Predicted Default Probabilities\n(Good separation = model is working well)',
             fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

print('A well-calibrated model pushes good applicants left (low prob) and bad applicants right (high prob).')

---
## Section 6 — SHAP Explainability

### Why SHAP?
Traditional models are "black boxes". SHAP (SHapley Additive exPlanations) opens them up by answering:
> *"How much did each feature contribute to THIS specific prediction?"*

SHAP is based on **Shapley values** from cooperative game theory. Each feature is treated as a "player" in a game, and its contribution is fairly attributed across all possible combinations.

### SHAP value meaning:
- **Positive SHAP** → pushes prediction toward BAD (higher default risk)
- **Negative SHAP** → pushes prediction toward GOOD (lower risk, toward approval)
- **Zero SHAP** → feature had no effect on this prediction

For tree-based models (XGBoost, Random Forest), we use **TreeSHAP** — an exact, fast algorithm that computes SHAP values in polynomial time.

In [ ]:
# ── 6.1 Compute SHAP values ───────────────────────────────────────────────

xgb_model = trained_models['XGBoost']

# Convert test set to DataFrame with feature names (required by SHAP for labels)
X_test_df = pd.DataFrame(X_test, columns=feature_names)

# TreeExplainer is exact and fast for tree-based models
explainer  = shap.TreeExplainer(xgb_model)
shap_values = explainer(X_test_df)

print(f'SHAP values computed')
print(f'  Shape: {shap_values.values.shape}  ({shap_values.values.shape[0]} applicants × {shap_values.values.shape[1]} features)')
print(f'  Base value (average prediction): {shap_values.base_values[0]:.4f}')
print(f'  This means the average applicant has a {1/(1+np.exp(-shap_values.base_values[0])):.1%} predicted default probability')

In [ ]:
# ── 6.2 Global Feature Importance — Bar Chart ──────────────────────────────
# Mean |SHAP| = average absolute contribution of each feature across all 150 test applicants
# Answers: "Which features matter most for credit risk decisions overall?"

mean_shap = np.abs(shap_values.values).mean(axis=0)
importance_df = pd.DataFrame({
    'Feature':    feature_names,
    'Mean |SHAP|': mean_shap
}).sort_values('Mean |SHAP|', ascending=False)

print('Top 10 Most Important Features (Global SHAP):')
print(importance_df.head(10).to_string(index=False))

In [ ]:
# ── 6.3 SHAP Global Bar Plot ──────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(9, 7))
imp_sorted = importance_df.sort_values('Mean |SHAP|', ascending=True)
bars = ax.barh(imp_sorted['Feature'], imp_sorted['Mean |SHAP|'],
               color='#e05c5c', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Mean |SHAP value| (average impact on risk score)', fontsize=11)
ax.set_title('Global Feature Importance\nXGBoost Credit Risk Model (Test Set)',
             fontsize=13, fontweight='bold', pad=15)

for bar, val in zip(bars, imp_sorted['Mean |SHAP|']):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=8, color='#333')

plt.tight_layout()
plt.show()

print('\nInterpretation:')
print('  Checking account is the most important feature by a large margin.')
print('  Having no/little checking account balance is the biggest risk signal.')
print('  Duration and Credit amount also matter significantly.')

In [ ]:
# ── 6.4 SHAP Beeswarm Plot ────────────────────────────────────────────────
# The most informative SHAP plot — shows BOTH importance AND direction
# Each dot = one applicant
# X-axis  = SHAP value  (right = higher default risk)
# Colour  = feature value (red = high, blue = low)

fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.beeswarm(shap_values, max_display=15, show=False)
plt.title('SHAP Beeswarm — All Test Applicants\nEach dot = 1 applicant | Colour = feature value (red=high, blue=low)',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('How to read this:')
print('  • Checking account: blue dots (low balance = no account) cluster on the RIGHT → increases risk')
print('  • Checking account: red dots (high balance = rich) cluster on the LEFT  → reduces risk')
print('  • Duration: red dots (long duration) on the RIGHT → longer loans = more risk')
print('  • Age: red dots (older) on the LEFT → older applicants are safer')

In [ ]:
# ── 6.5 Local Explanation — Waterfall for a HIGH-RISK applicant ────────────
# Answers: "Why was THIS specific applicant rejected?"

# Find the most convincingly bad applicant (true positive with highest risk score)
risk_scores = shap_values.values.sum(axis=1) + shap_values.base_values
predicted_bad = (1/(1+np.exp(-risk_scores))) > 0.5
true_bad = y_test.values == 1
true_positives = np.where(predicted_bad & true_bad)[0]
bad_idx = true_positives[np.argmax(risk_scores[true_positives])]

print(f'Selected applicant index: #{bad_idx}')
print(f'True label: BAD (will default)')
print(f'Predicted default probability: {1/(1+np.exp(-risk_scores[bad_idx])):.1%}')
print()
print('Raw feature values:')
for col in X_test_df.columns:
    print(f'  {col:<30}: {X_test_df.iloc[bad_idx][col]:.4f}')

In [ ]:
# ── 6.6 Waterfall plot — rejected applicant ────────────────────────────────

fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.waterfall(shap_values[bad_idx], max_display=15, show=False)
plt.title(f'Why Was Applicant #{bad_idx} Rejected?\n'
          f'Each bar shows one feature\'s contribution to the risk score',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

print('How to read the waterfall:')
print('  E[f(X)] = base value = average prediction for all applicants')
print('  Red bars (+) = features that increased risk')
print('  Blue bars (-) = features that reduced risk')
print('  f(X) = final prediction for this applicant')

In [ ]:
# ── 6.7 Waterfall plot — approved applicant (contrast) ────────────────────

predicted_good = (1/(1+np.exp(-risk_scores))) <= 0.5
true_good = y_test.values == 0
true_negatives = np.where(predicted_good & true_good)[0]
good_idx = true_negatives[np.argmin(risk_scores[true_negatives])]

print(f'Approved applicant index: #{good_idx}')
print(f'True label: GOOD')
print(f'Predicted default probability: {1/(1+np.exp(-risk_scores[good_idx])):.1%}')
print()

fig, ax = plt.subplots(figsize=(10, 7))
shap.plots.waterfall(shap_values[good_idx], max_display=15, show=False)
plt.title(f'Why Was Applicant #{good_idx} Approved?\n'
          f'Compare to the rejected applicant above',
          fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

In [ ]:
# ── 6.8 SHAP Dependence Plot ──────────────────────────────────────────────
# Shows how a single feature's value affects SHAP (risk contribution)
# Interaction colour = another feature that explains the vertical scatter

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, feat in zip(axes, ['Checking account', 'Duration']):
    feat_idx = feature_names.index(feat)
    shap.plots.scatter(shap_values[:, feat_idx], ax=ax, show=False)
    ax.set_title(f'SHAP Dependence: {feat}', fontweight='bold')
    ax.set_xlabel(feat)
    ax.set_ylabel(f'SHAP value for {feat}')

plt.suptitle('How Feature Value Affects Risk Contribution',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print('Dependence plot interpretation:')
print('  Checking account: 0=none → very high SHAP (+risk), 3=rich → negative SHAP (-risk)')
print('  Duration: longer durations → higher SHAP → more risk')

In [ ]:
# ── 6.9 Adverse Action Notice ──────────────────────────────────────────────
# Plain-English explanation — required by fair lending laws (e.g., ECOA in the US)

sv = shap_values.values[bad_idx]
factor_df = pd.DataFrame({
    'Feature': feature_names,
    'SHAP':    sv
}).sort_values('SHAP', ascending=False)

rejection_reasons = factor_df[factor_df['SHAP'] > 0].head(3)
approval_factors  = factor_df[factor_df['SHAP'] < 0].head(3)

print('='*55)
print(f'  ADVERSE ACTION NOTICE — Applicant #{bad_idx}')
print('='*55)
print('  Your application requires further review.')
print('  Principal reasons for elevated risk:\n')
for _, row in rejection_reasons.iterrows():
    print(f'  [✗] {row["Feature"]:<28}  (risk impact: +{row["SHAP"]:.4f})')
print('\n  Factors in your favour:\n')
for _, row in approval_factors.iterrows():
    print(f'  [✓] {row["Feature"]:<28}  (risk impact: {row["SHAP"]:.4f})')
print('='*55)
print()
print('This type of notice is legally required in many jurisdictions.')
print('SHAP makes it possible to generate it automatically for every applicant.')

---
## Section 7 — Live API Demo

The trained XGBoost model is served via a Flask REST API.  
You can also use the interactive web form at **http://127.0.0.1:5000/**

**Start the API first:** `python api/app.py`

In [ ]:
# ── 7.1 Health check ──────────────────────────────────────────────────────

import requests
import json

BASE_URL = 'http://127.0.0.1:5000'

try:
    r = requests.get(f'{BASE_URL}/health', timeout=3)
    print('Health check:')
    print(json.dumps(r.json(), indent=2))
except requests.exceptions.ConnectionError:
    print('API is not running.')
    print('Start it with: python api/app.py')
    print('Then re-run this cell.')

In [ ]:
# ── 7.2 High-risk applicant prediction ───────────────────────────────────

high_risk_applicant = {
    'Age':              22,
    'Sex':              'female',
    'Job':              0,
    'Housing':          'rent',
    'Saving accounts':  'none',
    'Checking account': 'none',
    'Credit amount':    15000,
    'Duration':         60,
    'Purpose':          'education'
}

try:
    r = requests.post(f'{BASE_URL}/predict', json=high_risk_applicant, timeout=5)
    result = r.json()
    print('=== HIGH-RISK APPLICANT ===')
    print(json.dumps(result, indent=2))
except:
    print('Start the API first: python api/app.py')

In [ ]:
# ── 7.3 Low-risk applicant prediction ────────────────────────────────────

low_risk_applicant = {
    'Age':              45,
    'Sex':              'male',
    'Job':              3,
    'Housing':          'own',
    'Saving accounts':  'rich',
    'Checking account': 'rich',
    'Credit amount':    1500,
    'Duration':         12,
    'Purpose':          'car'
}

try:
    r = requests.post(f'{BASE_URL}/predict', json=low_risk_applicant, timeout=5)
    result = r.json()
    print('=== LOW-RISK APPLICANT ===')
    print(json.dumps(result, indent=2))
except:
    print('Start the API first: python api/app.py')

---
## Summary

### What We Built

| Component | Detail |
|-----------|--------|
| Dataset | German Credit (1,000 applicants, 9 features) |
| Best Model | XGBoost — ROC-AUC 0.770, Gini 0.540 |
| Explainability | SHAP (TreeSHAP) — global and per-applicant explanations |
| API | Flask REST API + interactive web UI at `/` |
| Key Insight | Checking account balance is the #1 risk signal |

### Key Lessons

1. **Never use accuracy for imbalanced classification** — use ROC-AUC, F1, Recall(bad)
2. **Informative missingness** — NaN can carry signal (no account = higher risk)
3. **Fit preprocessors on train only** — prevents data leakage
4. **SHAP explains any individual prediction** — enables regulatory compliance
5. **class_weight / scale_pos_weight** — essential for imbalanced credit data

### Files Generated
- `models/xgboost.pkl` — trained XGBoost model
- `models/preprocessor.pkl` — fitted sklearn ColumnTransformer
- `models/feature_names.pkl` — list of 19 feature names
- `reports/shap_global_bar.png` — global feature importance
- `reports/shap_beeswarm.png` — summary of all SHAP values
- `reports/shap_waterfall_*.png` — per-applicant explanations
- `reports/model_comparison.csv` — metrics for all 3 models